<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/956_EAASv3_UnitTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This test suite is **very strong**. From a software engineering perspective—and especially for an **AI reliability agent**—this is exactly the level of coverage you want. I’ll review it the same way we’ve been reviewing the rest of your agents: **architecture coverage → test quality → gaps → small improvements**.

---

# 1. Big Picture: Test Coverage

Your tests cover **all four layers of the system**, which is excellent.

| Layer          | Tested | Notes                              |
| -------------- | ------ | ---------------------------------- |
| Data loading   | ✅      | catalog validation + run filtering |
| Scoring        | ✅      | scenario scoring + metrics rollup  |
| Trend analysis | ✅      | trigger history logic              |
| Reporting      | ✅      | executive report structure         |

That means this test suite verifies the **entire evaluation pipeline**:

```
scenario input
   ↓
scenario scoring
   ↓
run metrics
   ↓
trend detection
   ↓
executive report
```

That’s exactly what a **QA framework for an AI system** should test.

---

# 2. Scoring Tests

This section is **excellent**.

### paths_match

You correctly test:

```
identical
order mismatch
subset mismatch
empty lists
```

These cover the realistic failure modes.

---

### percentile

You test:

```
empty input
single value
multiple values
edge percentiles
```

This prevents a **very common percentile bug** (empty or single-value datasets).

Good defensive testing.

---

### score_scenario

These tests are particularly strong.

You validate the **failure chain detection**:

| Test                 | Purpose              |
| -------------------- | -------------------- |
| all correct          | success path         |
| classification error | earliest failure     |
| routing error        | second stage failure |
| outcome error        | final stage failure  |

This guarantees your **failure_type logic works correctly**.

Which is important because that drives:

```
diagnostics
root cause classification
report insights
```

---

# 3. Risk Logic Tests

You also verify the **business risk layer**.

Example:

```python
test_is_high_risk_failure_failed_high_business_risk
```

This ensures your system correctly interprets:

```
high risk + failure → high_risk_failure
```

This is critical for:

```
weighted failure scoring
risk reporting
executive alerts
```

---

# 4. Rollup Metrics Tests

These are excellent.

You test:

```
empty scenario set
multi-scenario aggregation
latency aggregation
pass rate math
```

Example:

```python
assert m.avg_latency_ms == 150.0
assert m.max_latency_ms == 200.0
```

This protects against **math errors in rollups**, which are very common in metrics pipelines.

---

# 5. Evaluation Score

You correctly test:

```python
score = calculate_evaluation_score(...)
assert 0.85 <= score <= 0.95
```

Using **ranges instead of exact values** is smart.

Why?

Floating point calculations + weights can change slightly over time.

Your test will not break unnecessarily.

---

# 6. End-to-End Scoring Utility

This is a **very important test**.

```
test_score_evaluation_run_minimal
```

This ensures the **top-level scoring interface works**.

This validates:

```
scenario scoring
run metrics creation
max_latency inclusion
failure type propagation
```

This protects the **API contract**.

---

# 7. Data Loading Tests

These are very well designed.

You use **pytest temp directories**:

```python
tmp_path
```

That is the correct way to test file-based utilities.

You correctly test two critical cases:

### Case 1

```
run_id exists
but no matching scenarios
```

Expected behavior:

```
error added to state
```

This is exactly what your v3.1 improvements introduced.

---

### Case 2

```
run_id matches scenario history
```

Expected behavior:

```
scenario_results_for_run populated
```

Great coverage.

---

# 8. Catalog Validation Tests

These tests verify your **data integrity guardrails**.

You check:

```
missing field
null field
required field constants
```

That protects against malformed scenario catalogs.

Which is important because catalogs are often edited manually.

---

# 9. Trend Analysis Tests

This test is very important:

```python
test_known_triggers_non_empty
```

This ensures your **trigger schema exists**.

You also test:

```
first run history generation
system_status determination
```

Which ensures your monitoring layer behaves correctly.

---

# 10. Reporting Tests

These are excellent.

You test two things:

### Structural validity

```
Verdict exists
Targets table exists
Scenario count exists
```

This ensures your **executive reporting format never breaks**.

---

### Diagnostics rendering

You verify:

```
Failure cause column
Scenario ID presence
classification_error rendering
```

This protects your **root cause reporting**.

---

# 11. One Subtle Strength

You used **minimal datasets**.

Example:

```
1 scenario
2 scenarios
small catalogs
```

This is the correct philosophy for unit tests.

Tests should be:

```
fast
small
deterministic
```

Your suite follows that perfectly.

---

# 12. One Small Bug I Noticed

Your report test uses:

```python
build_report_markdown(
    run_id="RUN_1",
    run_metrics=run_metrics,
    trigger_summary=trigger_summary,
    scenario_scores=scenario_scores,
    config=config,
)
```

But the earlier reporting code you showed did **not include `config` in the function signature**.

Earlier version:

```
build_report_markdown(
    run_id,
    run_metrics,
    trigger_summary,
    scenario_scores
)
```

So either:

* you **updated the function signature**, or
* the test will fail.

Just double-check those match.

---

# 13. Two Tests Worth Adding

Your coverage is already very good, but two tests would make it **nearly perfect**.

---

### Test 1 — Failure Sorting

Since you added **severity sorting**, test it.

Example:

```python
def test_report_sorts_failures_by_severity():
    scenario_scores = [
        {"scenario_id": "A", "severity_weight": 1.0, ...},
        {"scenario_id": "B", "severity_weight": 5.0, ...},
    ]

    md = build_report_markdown(...)

    assert md.index("B") < md.index("A")
```

This guarantees **highest severity appears first**.

---

### Test 2 — No Scenarios Case

Since you added special handling, test it.

Example:

```python
def test_report_no_scenarios():
    run_metrics = {"total_scenarios": 0, ...}

    md = build_report_markdown(...)

    assert "Total evaluated: 0" in md
```

This protects the **empty-run edge case**.

---

# 14. Overall Quality Assessment

If I were grading this as production QA coverage:

| Category              | Score |
| --------------------- | ----- |
| Architecture coverage | ⭐⭐⭐⭐⭐ |
| Edge cases            | ⭐⭐⭐⭐  |
| Data validation       | ⭐⭐⭐⭐⭐ |
| Metric correctness    | ⭐⭐⭐⭐⭐ |
| Reporting validation  | ⭐⭐⭐⭐  |

Overall:

**9 / 10 test coverage quality**

Which is **very strong for an AI agent project**.

Many ML systems ship with far less testing.

---

# 15. Strategic Insight

Your EaaS test suite is doing something **important beyond correctness**.

It is turning AI evaluation into something **deterministic and auditable**.

That is exactly what enterprise AI systems need.

Instead of:

```
AI worked
AI failed
```

You now have:

```
which scenario failed
why it failed
how severe it was
how it affects system risk
```

That’s a **true AI reliability layer**.

---

✅ **Conclusion**

This is a **high-quality test suite** and absolutely appropriate for the **EaaS v3 orchestrator**.

Only two small improvements would make it near perfect:

1. Add a **severity sorting test**
2. Add a **zero-scenario report test**




In [ ]:
"""Unit tests for EaaS v3 orchestrator utilities.

Test data loading, scoring, trend analysis, and reporting with minimal/mock data.
Run from project root: python -m pytest test_eaas_v3_utilities.py -v --tb=short
"""

import json
import sys
from pathlib import Path

import pytest

# Project root for imports
root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from config import EvalAsServiceConfig
from agents.eaas_v3.orchestrator.utilities.data_loading import (
    load_eaas_data,
    _validate_scenario_catalog,
    SCENARIO_CATALOG_REQUIRED_FIELDS,
)
from agents.eaas_v3.orchestrator.utilities.scoring import (
    paths_match,
    percentile,
    score_scenario,
    rollup_run_metrics,
    score_evaluation_run,
    is_high_risk_failure,
    calculate_evaluation_score,
    generate_trigger_flags,
    ScenarioScore,
    RunMetrics,
)
from agents.eaas_v3.orchestrator.utilities.trend_analysis import (
    update_trigger_history_with_run,
    KNOWN_TRIGGERS,
)
from agents.eaas_v3.orchestrator.utilities.reporting import build_report_markdown


# -----------------------------------------------------------------------------
# Scoring: paths_match
# -----------------------------------------------------------------------------
def test_paths_match_identical():
    assert paths_match(["a", "b"], ["a", "b"]) is True


def test_paths_match_different_order():
    assert paths_match(["a", "b"], ["b", "a"]) is False


def test_paths_match_subset():
    assert paths_match(["a", "b", "c"], ["a", "b"]) is False


def test_paths_match_empty():
    assert paths_match([], []) is True


# -----------------------------------------------------------------------------
# Scoring: percentile
# -----------------------------------------------------------------------------
def test_percentile_empty():
    assert percentile([], 0.95) == 0.0


def test_percentile_single():
    assert percentile([100.0], 0.5) == 100.0


def test_percentile_five_values():
    vals = [10.0, 20.0, 30.0, 40.0, 50.0]
    assert percentile(vals, 0.5) == 30.0
    assert percentile(vals, 0.0) == 10.0
    assert percentile(vals, 1.0) == 50.0


# -----------------------------------------------------------------------------
# Scoring: score_scenario and failure_type
# -----------------------------------------------------------------------------
def test_score_scenario_all_correct():
    s = score_scenario(
        run_id="RUN_1",
        scenario_id="SCN_1",
        expected_issue_type="x",
        predicted_issue_type="x",
        expected_resolution_path=["a"],
        predicted_resolution_path=["a"],
        expected_outcome="o",
        predicted_outcome="o",
        latency_ms=100.0,
        human_review_required=False,
    )
    assert s.issue_classification_correct is True
    assert s.resolution_path_correct is True
    assert s.outcome_correct is True
    assert s.failure_type == "passed"


def test_score_scenario_classification_error():
    s = score_scenario(
        run_id="RUN_1",
        scenario_id="SCN_1",
        expected_issue_type="x",
        predicted_issue_type="y",
        expected_resolution_path=["a"],
        predicted_resolution_path=["a"],
        expected_outcome="o",
        predicted_outcome="o",
        latency_ms=100.0,
        human_review_required=False,
    )
    assert s.issue_classification_correct is False
    assert s.failure_type == "classification_error"


def test_score_scenario_routing_error():
    s = score_scenario(
        run_id="RUN_1",
        scenario_id="SCN_1",
        expected_issue_type="x",
        predicted_issue_type="x",
        expected_resolution_path=["a", "b"],
        predicted_resolution_path=["a"],
        expected_outcome="o",
        predicted_outcome="o",
        latency_ms=100.0,
        human_review_required=False,
    )
    assert s.resolution_path_correct is False
    assert s.failure_type == "routing_error"


def test_score_scenario_outcome_error():
    s = score_scenario(
        run_id="RUN_1",
        scenario_id="SCN_1",
        expected_issue_type="x",
        predicted_issue_type="x",
        expected_resolution_path=["a"],
        predicted_resolution_path=["a"],
        expected_outcome="o1",
        predicted_outcome="o2",
        latency_ms=100.0,
        human_review_required=False,
    )
    assert s.outcome_correct is False
    assert s.failure_type == "outcome_error"


# -----------------------------------------------------------------------------
# Scoring: is_high_risk_failure, rollup, evaluation_score
# -----------------------------------------------------------------------------
def test_is_high_risk_failure_passed_not_high_risk():
    s = ScenarioScore(
        scenario_id="S",
        run_id="R",
        issue_classification_correct=True,
        resolution_path_correct=True,
        outcome_correct=True,
        latency_ms=0.0,
        human_review_required=False,
        business_risk="low",
        severity_weight=1.0,
    )
    assert is_high_risk_failure(s) is False


def test_is_high_risk_failure_failed_high_business_risk():
    s = ScenarioScore(
        scenario_id="S",
        run_id="R",
        issue_classification_correct=False,
        resolution_path_correct=True,
        outcome_correct=True,
        latency_ms=0.0,
        human_review_required=False,
        business_risk="high",
        severity_weight=1.0,
    )
    assert is_high_risk_failure(s) is True


def test_rollup_run_metrics_empty():
    m = rollup_run_metrics("RUN_1", [])
    assert m.run_id == "RUN_1"
    assert m.total_scenarios == 0
    assert m.scenarios_passed == 0
    assert m.overall_pass_rate == 0.0
    assert m.max_latency_ms == 0.0


def test_rollup_run_metrics_two_scenarios():
    s1 = score_scenario(
        run_id="RUN_1",
        scenario_id="S1",
        expected_issue_type="x",
        predicted_issue_type="x",
        expected_resolution_path=[],
        predicted_resolution_path=[],
        expected_outcome="o",
        predicted_outcome="o",
        latency_ms=100.0,
        human_review_required=False,
    )
    s2 = score_scenario(
        run_id="RUN_1",
        scenario_id="S2",
        expected_issue_type="x",
        predicted_issue_type="y",
        expected_resolution_path=[],
        predicted_resolution_path=[],
        expected_outcome="o",
        predicted_outcome="o",
        latency_ms=200.0,
        human_review_required=False,
    )
    m = rollup_run_metrics("RUN_1", [s1, s2])
    assert m.total_scenarios == 2
    assert m.scenarios_passed == 1
    assert m.overall_pass_rate == 0.5
    assert m.issue_classification_accuracy == 0.5
    assert m.max_latency_ms == 200.0
    assert m.avg_latency_ms == 150.0


def test_calculate_evaluation_score():
    score = calculate_evaluation_score(
        outcome_accuracy=0.9,
        resolution_path_accuracy=0.9,
        issue_classification_accuracy=0.9,
        human_review_rate=0.1,
    )
    assert 0.85 <= score <= 0.95


# -----------------------------------------------------------------------------
# Scoring: score_evaluation_run (end-to-end utility)
# -----------------------------------------------------------------------------
def test_score_evaluation_run_minimal():
    scenario_inputs = [
        {
            "run_id": "RUN_1",
            "scenario_id": "S1",
            "expected_issue_type": "x",
            "predicted_issue_type": "x",
            "expected_resolution_path": ["a"],
            "predicted_resolution_path": ["a"],
            "expected_outcome": "o",
            "predicted_outcome": "o",
            "latency_ms": 100.0,
            "human_review_required": False,
        },
    ]
    out = score_evaluation_run("RUN_1", scenario_inputs, previous_run_metrics=None)
    assert "scenario_scores" in out
    assert "run_metrics" in out
    assert len(out["scenario_scores"]) == 1
    assert out["scenario_scores"][0]["failure_type"] == "passed"
    assert out["run_metrics"]["total_scenarios"] == 1
    assert out["run_metrics"]["scenarios_passed"] == 1
    assert "max_latency_ms" in out["run_metrics"]


# -----------------------------------------------------------------------------
# Data loading: catalog validation
# -----------------------------------------------------------------------------
def test_validate_scenario_catalog_valid():
    catalog = [
        {"scenario_id": "S1", "severity_weight": 1, "business_risk": "low"},
    ]
    assert _validate_scenario_catalog(catalog) == []


def test_validate_scenario_catalog_missing_field():
    catalog = [
        {"scenario_id": "S1", "severity_weight": 1},
    ]
    errs = _validate_scenario_catalog(catalog)
    assert any("business_risk" in e for e in errs)


def test_validate_scenario_catalog_null_field():
    catalog = [
        {"scenario_id": "S1", "severity_weight": None, "business_risk": "low"},
    ]
    errs = _validate_scenario_catalog(catalog)
    assert any("severity_weight" in e for e in errs)


def test_scenario_catalog_required_fields():
    assert "scenario_id" in SCENARIO_CATALOG_REQUIRED_FIELDS
    assert "severity_weight" in SCENARIO_CATALOG_REQUIRED_FIELDS
    assert "business_risk" in SCENARIO_CATALOG_REQUIRED_FIELDS


# -----------------------------------------------------------------------------
# Data loading: load_eaas_data with temp dir
# -----------------------------------------------------------------------------
def test_load_eaas_data_no_scenarios_for_run_adds_error(tmp_path):
    """When run_id is set but no scenarios match, state gets an error."""
    (tmp_path / "scenario_results_history.json").write_text(json.dumps([]))
    (tmp_path / "trigger_history.json").write_text(json.dumps([]))
    (tmp_path / "scenario_catalog_enriched.json").write_text(json.dumps([
        {"scenario_id": "S1", "severity_weight": 1, "business_risk": "low"},
    ]))
    config = EvalAsServiceConfig()
    config.data_dir = str(tmp_path)
    state = {"project_root": str(tmp_path), "run_id": "RUN_NONE", "errors": []}
    out = load_eaas_data(state=state, config=config)
    assert out["run_id"] == "RUN_NONE"
    assert out["scenario_results_for_run"] == []
    assert any("No scenarios found" in e for e in out.get("errors", []))


def test_load_eaas_data_with_matching_run(tmp_path):
    history = [
        {"run_id": "RUN_A", "scenario_id": "S1", "target_version": "v1"},
    ]
    (tmp_path / "scenario_results_history.json").write_text(json.dumps(history))
    (tmp_path / "trigger_history.json").write_text(json.dumps([]))
    (tmp_path / "scenario_catalog_enriched.json").write_text(json.dumps([
        {"scenario_id": "S1", "severity_weight": 1, "business_risk": "low"},
    ]))
    config = EvalAsServiceConfig()
    config.data_dir = str(tmp_path)
    state = {"project_root": str(tmp_path), "run_id": "RUN_A", "errors": []}
    out = load_eaas_data(state=state, config=config)
    assert len(out["scenario_results_for_run"]) == 1
    assert out["scenario_results_for_run"][0]["scenario_id"] == "S1"


# -----------------------------------------------------------------------------
# Trend analysis: KNOWN_TRIGGERS and update_trigger_history_with_run
# -----------------------------------------------------------------------------
def test_known_triggers_non_empty():
    assert len(KNOWN_TRIGGERS) >= 5
    assert "regression_trigger" in KNOWN_TRIGGERS
    assert "critical_risk_trigger" in KNOWN_TRIGGERS


def test_update_trigger_history_with_run_first_run():
    run_metrics = {
        "run_id": "RUN_1",
        "total_scenarios": 2,
        "scenarios_passed": 2,
        "overall_pass_rate": 1.0,
        "issue_classification_accuracy": 1.0,
        "resolution_path_accuracy": 1.0,
        "outcome_accuracy": 1.0,
        "high_risk_failures": 0,
        "human_review_rate": 0.0,
        "avg_latency_ms": 100.0,
        "p95_latency_ms": 100.0,
        "max_latency_ms": 100.0,
        "hallucination_rate": 0.0,
        "policy_violation_rate": 0.0,
        "weighted_failure_rate": 0.0,
        "evaluation_score": 0.9,
        "regression_flags": [],
        "improvement_flags": [],
        "trigger_flags": [],
    }
    entry, history = update_trigger_history_with_run(
        run_id="RUN_1",
        run_metrics=run_metrics,
        trigger_history=[],
        scoring_version="eaas_v3.0",
    )
    assert entry["run_id"] == "RUN_1"
    assert entry["system_status"] == "healthy"
    assert len(history) == 1


# -----------------------------------------------------------------------------
# Reporting: build_report_markdown
# -----------------------------------------------------------------------------
def test_build_report_markdown_contains_verdict_and_sections():
    config = EvalAsServiceConfig()
    run_metrics = {
        "evaluation_score": 0.8,
        "overall_pass_rate": 0.8,
        "outcome_accuracy": 0.8,
        "total_scenarios": 5,
        "scenarios_passed": 4,
        "high_risk_failures": 0,
        "trigger_flags": [],
        "regression_flags": [],
        "improvement_flags": [],
    }
    trigger_summary = {"system_status": "healthy"}
    scenario_scores = []
    md = build_report_markdown(
        run_id="RUN_1",
        run_metrics=run_metrics,
        trigger_summary=trigger_summary,
        scenario_scores=scenario_scores,
        config=config,
    )
    assert "Verdict:" in md
    assert "GREEN" in md or "WARNING" in md or "CRITICAL" in md
    assert "Targets vs actuals" in md
    assert "Scenario count" in md
    assert "Total evaluated" in md


def test_build_report_markdown_failure_diagnostics():
    config = EvalAsServiceConfig()
    run_metrics = {
        "evaluation_score": 0.7,
        "overall_pass_rate": 0.5,
        "outcome_accuracy": 0.5,
        "total_scenarios": 2,
        "scenarios_passed": 1,
        "high_risk_failures": 1,
        "trigger_flags": ["weighted_failure_trigger"],
        "regression_flags": [],
        "improvement_flags": [],
    }
    trigger_summary = {"system_status": "warning"}
    scenario_scores = [
        {
            "scenario_id": "SCN_004",
            "issue_classification_correct": False,
            "resolution_path_correct": False,
            "outcome_correct": False,
            "business_risk": "high",
            "severity_weight": 4.0,
            "failure_type": "classification_error",
        },
    ]
    md = build_report_markdown(
        run_id="RUN_1",
        run_metrics=run_metrics,
        trigger_summary=trigger_summary,
        scenario_scores=scenario_scores,
        config=config,
    )
    assert "Scenario Diagnostics" in md
    assert "Failure cause" in md
    assert "classification_error" in md
    assert "SCN_004" in md


def test_generate_trigger_flags_critical_risk():
    m = RunMetrics(
        run_id="R",
        total_scenarios=1,
        scenarios_passed=0,
        overall_pass_rate=0.0,
        issue_classification_accuracy=0.0,
        resolution_path_accuracy=0.0,
        outcome_accuracy=0.0,
        high_risk_failures=3,
        human_review_rate=0.0,
        avg_latency_ms=500.0,
        p95_latency_ms=500.0,
        max_latency_ms=500.0,
        hallucination_rate=0.0,
        policy_violation_rate=0.0,
        weighted_failure_rate=0.5,
        evaluation_score=0.5,
        regression_flags=[],
        improvement_flags=[],
        trigger_flags=[],
    )
    flags = generate_trigger_flags(m)
    assert "critical_risk_trigger" in flags
